
# ARC-v0.18.1 — E5 Cross-Encoder Signed-Direction Audit

**Purpose.** Perform a post-hoc construct-validity audit of the completed ARC-v0.18 E5 cross-encoder replication.

ARC-v0.18 established a **PARTIAL_REPLICATION** under `intfloat/e5-small-v2`: aggregate H1/H2/H3 directions remained positive, amplification remained a minority regime overall, and feedback-gain incidence increased from \(\alpha=0.1\) to \(\alpha=0.7\), but stable/null trajectories were no longer a majority at the primary threshold.

This audit asks a new, narrower question:

> **Among E5 validation query-policy events classified as absolute amplification, is the divergence predominantly harmful to the lower-fidelity PQ32 trajectory, beneficial to it, or mixed?**

No retrieval is rerun. No encoder, index, feedback, split, threshold, or policy parameter is changed.

## Sign convention

For each round \(t\),

\[
G_t = u_{\mathrm{SQ8}}(t)-u_{\mathrm{PQ32}}(t).
\]

Therefore:

- \(G_T>0\): the higher-fidelity SQ8 trajectory ends with higher utility, so the approximation-induced divergence is **directionally harmful to PQ32**;
- \(G_T<0\): the PQ32 trajectory ends with higher utility, so the divergence is **beneficial to PQ32**;
- \(G_T=0\): tied / unresolved final direction.

The original absolute amplification endpoint is

\[
H3_{\mathrm{abs}}
=
\operatorname{slope}\left(|G_t|\right).
\]

The audit also reports

\[
H3_{\mathrm{signed}}
=
\operatorname{slope}(G_t)
\]

and

\[
\Delta G = G_T-G_0.
\]

## Primary population

Untouched ARC-v0.18 validation query-policy events satisfying

\[
H3_{\mathrm{abs}} > 0.002.
\]

The primary signed conclusion is classified as:

- **SIGNED_HARM_PRESERVED** if the query-cluster bootstrap 95% CI lower bound for \(P(G_T>0\mid H3_{\mathrm{abs}}>0.002)\) is \(>0.5\);
- **MIXED_DIRECTION** if the interval includes 0.5;
- **BENEFICIAL_DOMINANT** if the interval upper bound is \(<0.5\).

This post-hoc gate does **not** alter the original ARC-v0.18 `PARTIAL_REPLICATION` claim gate.


In [ ]:

# ============================================================
# Cell 1 — Imports / Drive / frozen source run
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math

import numpy as np
import pandas as pd

from google.colab import drive

SEED = 20260819
BOOTSTRAP_REPS = 10_000

EPS_PRIMARY = 0.002
EPS_SWEEP = [0.0, 0.001, 0.002, 0.005, 0.01]

EXPECTED_VAL_QUERIES = 3316
EXPECTED_CONFIGS = 44
EXPECTED_ROUNDS = 5

SOURCE_REPORT_SHA256 = (
    "e6e4aaabe8ef6feabe1c702c1e60b356"
    "6f7d1c45f02fb374a58532488b099221"
)

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), "Google Drive mount failed."

ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"

SOURCE_RUN = (
    ARC_ROOT
    / "cross-encoder-fever-replication-v018"
    / "20260819-015645"
)

assert SOURCE_RUN.is_dir(), SOURCE_RUN

print("Source run:", SOURCE_RUN)
print("Bootstrap reps:", BOOTSTRAP_REPS)


In [ ]:

# ============================================================
# Cell 2 — Verify sealed ARC-v0.18 source integrity
# ============================================================

def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

SOURCE_REPORT = (
    SOURCE_RUN
    / "v018_cross_encoder_replication_report.json"
)

assert SOURCE_REPORT.is_file(), SOURCE_REPORT

source_report = json.loads(
    SOURCE_REPORT.read_text(encoding="utf-8")
)

actual_sha = sha256_file(SOURCE_REPORT)

print("Source report SHA-256:", actual_sha)
print("Source claim gate:", source_report["claim_gate"])
print("Source test_accessed:", source_report["test_accessed"])

assert actual_sha == SOURCE_REPORT_SHA256, (
    "ARC-v0.18 source report hash mismatch. "
    "Do not continue until provenance is resolved."
)

assert source_report["claim_gate"] == "PARTIAL_REPLICATION"
assert source_report["test_accessed"] is False
assert source_report["test_relevance_accessed"] is False

print("ARC-v0.18 SOURCE INTEGRITY — PASS")


In [ ]:

# ============================================================
# Cell 3 — Create and seal v0.18.1 audit protocol
#
# This is explicitly post-hoc with respect to ARC-v0.18.
# The event-level signed taxonomy is not inspected before this
# protocol file is written in the current audit run.
# ============================================================

AUDIT_ROOT = (
    ARC_ROOT
    / "cross-encoder-signed-direction-audit-v0181"
)
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = AUDIT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

PROTOCOL = {
    "status":
        "ARC_V0181_SIGNED_DIRECTION_AUDIT_PROTOCOL_SEALED",

    "audit_type":
        "post_hoc_construct_validity",

    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "source_v018_run":
        str(SOURCE_RUN),

    "source_v018_report_sha256":
        SOURCE_REPORT_SHA256,

    "population":
        "ARC-v0.18 untouched validation query-policy events",

    "primary_amplification_threshold":
        EPS_PRIMARY,

    "threshold_sensitivity":
        EPS_SWEEP,

    "sign_convention":
        "G_t = utility_SQ8(t) - utility_PQ32(t)",

    "primary_signed_endpoint":
        "P(G_T > 0 | H3_abs > 0.002)",

    "secondary_signed_endpoints": [
        "P(H3_signed > 0 | H3_abs > 0.002)",
        "P(delta_G > 0 | H3_abs > 0.002)",
    ],

    "uncertainty":
        {
            "method": "query-cluster bootstrap",
            "replicates": BOOTSTRAP_REPS,
            "seed": SEED,
        },

    "claim_gate": {
        "SIGNED_HARM_PRESERVED":
            "95% CI lower bound for primary harmful fraction > 0.5",

        "MIXED_DIRECTION":
            "95% CI contains 0.5",

        "BENEFICIAL_DOMINANT":
            "95% CI upper bound < 0.5",
    },

    "test_accessed": False,
    "test_relevance_accessed": False,
}

PROTOCOL_PATH = OUT / "v0181_signed_direction_protocol.json"

PROTOCOL_PATH.write_text(
    json.dumps(
        PROTOCOL,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

PROTOCOL_SHA = sha256_file(PROTOCOL_PATH)

(
    OUT
    / "V0181_PROTOCOL_SHA256.txt"
).write_text(
    f"{PROTOCOL_SHA}  {PROTOCOL_PATH.name}\n",
    encoding="utf-8",
)

print("Audit output:", OUT)
print("Protocol SHA-256:", PROTOCOL_SHA)
print("V0.18.1 PROTOCOL SEALED — PASS")


In [ ]:

# ============================================================
# Cell 4 — Load all 44 untouched-validation trajectories
# ============================================================

VAL_DIR = SOURCE_RUN / "validation"
assert VAL_DIR.is_dir(), VAL_DIR

val_files = sorted(VAL_DIR.glob("validation-*.parquet"))

print("Validation files:", len(val_files))

assert len(val_files) == EXPECTED_CONFIGS, (
    f"Expected {EXPECTED_CONFIGS} validation configs, "
    f"found {len(val_files)}."
)

required = {
    "query_id",
    "iteration",
    "method",
    "alpha",
    "k",
    "temperature",
    "config_key",
    "utility_low",
    "utility_high",
    "signed_utility_gap",
    "abs_utility_gap",
}

frames = []

for i, path in enumerate(val_files, 1):
    df = pd.read_parquet(path)

    assert required.issubset(df.columns), (
        path.name,
        sorted(set(required) - set(df.columns)),
    )

    assert df["query_id"].nunique() == EXPECTED_VAL_QUERIES
    assert df["iteration"].nunique() == EXPECTED_ROUNDS

    expected_rows = EXPECTED_VAL_QUERIES * EXPECTED_ROUNDS
    assert len(df) == expected_rows, (
        path.name,
        len(df),
        expected_rows,
    )

    frames.append(df[list(required)].copy())

    if i in [1, 10, 20, 30, 40, 44]:
        print(
            f"[{i:02d}/44]",
            path.name,
            df.shape,
        )

traj = pd.concat(frames, ignore_index=True)

assert traj["config_key"].nunique() == EXPECTED_CONFIGS
assert traj["query_id"].nunique() == EXPECTED_VAL_QUERIES

print()
print("Trajectory rows:", len(traj))
print(
    "Expected:",
    EXPECTED_CONFIGS
    * EXPECTED_VAL_QUERIES
    * EXPECTED_ROUNDS,
)

print("VALIDATION TRAJECTORY LOAD — PASS")


In [ ]:

# ============================================================
# Cell 5 — Reconstruct exact signed/absolute event endpoints
# ============================================================

group_cols = [
    "query_id",
    "method",
    "alpha",
    "k",
    "temperature",
    "config_key",
]

rows = []

for keys, g in traj.groupby(
    group_cols,
    dropna=False,
    sort=False,
):
    g = g.sort_values("iteration")

    x = g["iteration"].to_numpy(np.float64)
    abs_gap = g["abs_utility_gap"].to_numpy(np.float64)
    signed_gap = g["signed_utility_gap"].to_numpy(np.float64)

    row = dict(zip(group_cols, keys))

    row.update({
        "H3_abs":
            float(np.polyfit(x, abs_gap, 1)[0]),

        "H3_signed":
            float(np.polyfit(x, signed_gap, 1)[0]),

        "G0":
            float(signed_gap[0]),

        "GT":
            float(signed_gap[-1]),

        "delta_G":
            float(signed_gap[-1] - signed_gap[0]),

        "abs_G0":
            float(abs_gap[0]),

        "abs_GT":
            float(abs_gap[-1]),
    })

    rows.append(row)

events = pd.DataFrame(rows)

expected_events = (
    EXPECTED_CONFIGS
    * EXPECTED_VAL_QUERIES
)

assert len(events) == expected_events
assert events["query_id"].nunique() == EXPECTED_VAL_QUERIES
assert events["config_key"].nunique() == EXPECTED_CONFIGS

events.to_parquet(
    OUT / "v0181_all_validation_signed_events.parquet",
    index=False,
)

print("Event rows:", len(events))
print("SIGNED ENDPOINT RECONSTRUCTION — PASS")


In [ ]:

# ============================================================
# Cell 6 — Exact parity with persisted ARC-v0.18 endpoints
# ============================================================

SOURCE_ENDPOINTS = (
    SOURCE_RUN
    / "v018_validation_endpoints.parquet"
)

assert SOURCE_ENDPOINTS.is_file(), SOURCE_ENDPOINTS

sealed_ep = pd.read_parquet(SOURCE_ENDPOINTS)

key = [
    "query_id",
    "config_key",
]

check = events.merge(
    sealed_ep[
        [
            "query_id",
            "config_key",
            "H3_abs_slope",
            "H3_signed_slope",
            "final_signed_gap",
        ]
    ],
    on=key,
    how="inner",
    validate="one_to_one",
)

assert len(check) == len(events)

diff_abs = np.abs(
    check["H3_abs"]
    -
    check["H3_abs_slope"]
)

diff_signed = np.abs(
    check["H3_signed"]
    -
    check["H3_signed_slope"]
)

diff_final = np.abs(
    check["GT"]
    -
    check["final_signed_gap"]
)

print("max |H3_abs replay - sealed|   :", diff_abs.max())
print("max |H3_signed replay - sealed|:", diff_signed.max())
print("max |GT replay - sealed|       :", diff_final.max())

assert diff_abs.max() <= 1e-12
assert diff_signed.max() <= 1e-12
assert diff_final.max() <= 1e-12

print("SIGNED REPLAY PARITY — PASS")


In [ ]:

# ============================================================
# Cell 7 — Primary amplification population and signed taxonomy
# ============================================================

TOL = 1e-12

amp = events[
    events["H3_abs"] > EPS_PRIMARY
].copy()

amp["final_direction"] = np.select(
    [
        amp["GT"] > TOL,
        amp["GT"] < -TOL,
    ],
    [
        "harmful_to_PQ32",
        "beneficial_to_PQ32",
    ],
    default="tied",
)

amp["signed_slope_direction"] = np.select(
    [
        amp["H3_signed"] > TOL,
        amp["H3_signed"] < -TOL,
    ],
    [
        "positive",
        "negative",
    ],
    default="tied",
)

amp["delta_direction"] = np.select(
    [
        amp["delta_G"] > TOL,
        amp["delta_G"] < -TOL,
    ],
    [
        "positive",
        "negative",
    ],
    default="tied",
)

coverage = len(amp) / len(events)

taxonomy = (
    amp["final_direction"]
    .value_counts(dropna=False)
    .rename_axis("signed_outcome")
    .reset_index(name="count")
)

taxonomy["fraction"] = (
    taxonomy["count"]
    / len(amp)
)

display(taxonomy)

print()
print("Total validation events :", len(events))
print("Amplification events    :", len(amp))
print("Amplification fraction  :", coverage)
print("Unique queries in amp   :", amp["query_id"].nunique())

amp.to_parquet(
    OUT / "v0181_primary_amplification_signed_events.parquet",
    index=False,
)

taxonomy.to_csv(
    OUT / "v0181_primary_signed_taxonomy.csv",
    index=False,
)

print("PRIMARY SIGNED TAXONOMY — COMPLETE")


In [ ]:

# ============================================================
# Cell 8 — Query-cluster bootstrap for signed fractions
# ============================================================

def cluster_bootstrap_fraction(
    df,
    positive_col,
    reps=BOOTSTRAP_REPS,
    seed=SEED,
):
    # Aggregate event numerator/denominator within each query,
    # then resample queries (clusters) with replacement.
    q = (
        df.groupby("query_id", as_index=False)
        .agg(
            numerator=(positive_col, "sum"),
            denominator=(positive_col, "size"),
        )
    )

    numer = q["numerator"].to_numpy(np.float64)
    denom = q["denominator"].to_numpy(np.float64)

    estimate = float(numer.sum() / denom.sum())

    rng = np.random.default_rng(seed)
    n = len(q)

    boots = np.empty(reps, dtype=np.float64)

    for b in range(reps):
        idx = rng.integers(
            0,
            n,
            size=n,
        )

        boots[b] = (
            numer[idx].sum()
            /
            denom[idx].sum()
        )

    lo, hi = np.quantile(
        boots,
        [0.025, 0.975],
    )

    return {
        "estimate": estimate,
        "ci_low": float(lo),
        "ci_high": float(hi),
        "bootstrap_reps": reps,
        "clusters": n,
    }

amp["is_harmful_final"] = (
    amp["GT"] > TOL
).astype(int)

amp["is_positive_signed_slope"] = (
    amp["H3_signed"] > TOL
).astype(int)

amp["is_positive_delta_G"] = (
    amp["delta_G"] > TOL
).astype(int)

primary_ci = cluster_bootstrap_fraction(
    amp,
    "is_harmful_final",
    seed=SEED + 1,
)

signed_slope_ci = cluster_bootstrap_fraction(
    amp,
    "is_positive_signed_slope",
    seed=SEED + 2,
)

delta_ci = cluster_bootstrap_fraction(
    amp,
    "is_positive_delta_G",
    seed=SEED + 3,
)

ci_table = pd.DataFrame([
    {
        "metric":
            "P(G_T > 0 | H3_abs > 0.002)",
        **primary_ci,
    },
    {
        "metric":
            "P(H3_signed > 0 | H3_abs > 0.002)",
        **signed_slope_ci,
    },
    {
        "metric":
            "P(delta_G > 0 | H3_abs > 0.002)",
        **delta_ci,
    },
])

display(ci_table)

ci_table.to_csv(
    OUT / "v0181_query_cluster_bootstrap_signed_estimates.csv",
    index=False,
)

print("QUERY-CLUSTER BOOTSTRAP — COMPLETE")


In [ ]:

# ============================================================
# Cell 9 — Method × alpha signed-harm strata
# ============================================================

strata = (
    amp.groupby(
        ["method", "alpha"],
        as_index=False,
    )
    .agg(
        events=("query_id", "size"),
        unique_queries=("query_id", "nunique"),
        harmful_fraction=("is_harmful_final", "mean"),
        positive_signed_slope_fraction=("is_positive_signed_slope", "mean"),
        positive_delta_G_fraction=("is_positive_delta_G", "mean"),
        mean_H3_abs=("H3_abs", "mean"),
        mean_H3_signed=("H3_signed", "mean"),
        mean_GT=("GT", "mean"),
    )
    .sort_values(
        ["method", "alpha"]
    )
)

display(strata)

strata.to_csv(
    OUT / "v0181_method_alpha_signed_strata.csv",
    index=False,
)

print(
    "Harmful-fraction range:",
    strata["harmful_fraction"].min(),
    "to",
    strata["harmful_fraction"].max(),
)

print("METHOD × ALPHA STRATA — COMPLETE")


In [ ]:

# ============================================================
# Cell 10 — Signed-harm sensitivity across epsilon thresholds
#
# Unlike the earlier BGE repair, v0.18 stores signed trajectories
# for all validation events, so all requested epsilon values are
# directly identifiable.
# ============================================================

sens_rows = []

for eps in EPS_SWEEP:
    d = events[
        events["H3_abs"] > eps
    ].copy()

    harmful = float(
        (d["GT"] > TOL).mean()
    )

    beneficial = float(
        (d["GT"] < -TOL).mean()
    )

    tied = float(
        (
            np.abs(
                d["GT"]
            )
            <= TOL
        ).mean()
    )

    sens_rows.append({
        "epsilon": eps,
        "events": len(d),
        "event_fraction": len(d) / len(events),
        "harmful_final_fraction": harmful,
        "beneficial_final_fraction": beneficial,
        "tied_final_fraction": tied,
        "positive_signed_slope_fraction":
            float(
                (d["H3_signed"] > TOL).mean()
            ),
        "positive_delta_G_fraction":
            float(
                (d["delta_G"] > TOL).mean()
            ),
    })

sensitivity = pd.DataFrame(sens_rows)

display(sensitivity)

sensitivity.to_csv(
    OUT / "v0181_signed_threshold_sensitivity.csv",
    index=False,
)

print("SIGNED THRESHOLD SENSITIVITY — COMPLETE")


In [ ]:

# ============================================================
# Cell 11 — Query-level concentration of harmful amplification
# ============================================================

per_query = (
    amp.groupby(
        "query_id",
        as_index=False,
    )
    .agg(
        amplification_events=("query_id", "size"),
        harmful_events=("is_harmful_final", "sum"),
        harmful_fraction=("is_harmful_final", "mean"),
        mean_H3_abs=("H3_abs", "mean"),
        mean_GT=("GT", "mean"),
    )
)

query_summary = {
    "queries_with_any_amplification":
        int(len(per_query)),

    "fraction_validation_queries_with_any_amplification":
        float(
            len(per_query)
            / EXPECTED_VAL_QUERIES
        ),

    "queries_with_all_amplification_events_harmful":
        int(
            (
                per_query[
                    "harmful_fraction"
                ]
                == 1.0
            ).sum()
        ),

    "queries_with_no_harmful_amplification_event":
        int(
            (
                per_query[
                    "harmful_fraction"
                ]
                == 0.0
            ).sum()
        ),

    "median_harmful_fraction_among_affected_queries":
        float(
            per_query[
                "harmful_fraction"
            ].median()
        ),
}

print(
    json.dumps(
        query_summary,
        indent=2,
    )
)

per_query.to_parquet(
    OUT / "v0181_per_query_signed_concentration.parquet",
    index=False,
)

print("QUERY-LEVEL CONCENTRATION — COMPLETE")


In [ ]:

# ============================================================
# Cell 12 — Frozen post-hoc signed-direction claim gate
# ============================================================

lo = primary_ci["ci_low"]
hi = primary_ci["ci_high"]

if lo > 0.5:
    CLAIM_GATE = "SIGNED_HARM_PRESERVED"
elif hi < 0.5:
    CLAIM_GATE = "BENEFICIAL_DOMINANT"
else:
    CLAIM_GATE = "MIXED_DIRECTION"

print("=" * 90)
print("ARC-v0.18.1 SIGNED-DIRECTION CLAIM GATE")
print("=" * 90)
print(
    "Primary harmful fraction:",
    primary_ci["estimate"],
)
print(
    "95% query-cluster bootstrap CI:",
    [
        primary_ci["ci_low"],
        primary_ci["ci_high"],
    ],
)
print(
    "Primary amplification events:",
    len(amp),
)
print(
    "Claim gate:",
    CLAIM_GATE,
)
print("=" * 90)


In [ ]:

# ============================================================
# Cell 13 — Seal final ARC-v0.18.1 report
# ============================================================

report = {
    "status":
        "ARC_V0181_CROSS_ENCODER_SIGNED_DIRECTION_AUDIT_COMPLETE",

    "audit_type":
        "post_hoc_construct_validity",

    "protocol_sha256":
        PROTOCOL_SHA,

    "source_v018_run":
        str(SOURCE_RUN),

    "source_v018_report_sha256":
        SOURCE_REPORT_SHA256,

    "source_v018_claim_gate":
        source_report["claim_gate"],

    "primary_threshold":
        EPS_PRIMARY,

    "validation_event_count":
        int(len(events)),

    "primary_amplification_event_count":
        int(len(amp)),

    "primary_amplification_fraction":
        float(len(amp) / len(events)),

    "signed_taxonomy":
        taxonomy.to_dict("records"),

    "primary_harmful_fraction":
        primary_ci,

    "positive_signed_slope_fraction":
        signed_slope_ci,

    "positive_delta_G_fraction":
        delta_ci,

    "method_alpha_strata":
        strata.to_dict("records"),

    "threshold_sensitivity":
        sensitivity.to_dict("records"),

    "query_level_concentration":
        query_summary,

    "claim_gate":
        CLAIM_GATE,

    "test_accessed":
        False,

    "test_relevance_accessed":
        False,

    "interpretation_constraints": [
        (
            "ARC-v0.18.1 is post-hoc with respect to the completed "
            "ARC-v0.18 cross-encoder replication."
        ),
        (
            "The audit does not alter the ARC-v0.18 "
            "PARTIAL_REPLICATION claim gate."
        ),
        (
            "Positive G_T means SQ8 has higher final nDCG@10 than "
            "PQ32 and is interpreted as directionally harmful to "
            "the lower-fidelity PQ32 trajectory."
        ),
        (
            "The analysis is conditional on absolute amplification "
            "under the stated epsilon threshold."
        ),
        (
            "No retrieval is rerun and no FEVER test outcomes are used."
        ),
    ],

    "completed_at_utc":
        datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH = (
    OUT
    / "v0181_cross_encoder_signed_direction_report.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

REPORT_SHA = sha256_file(REPORT_PATH)

(
    OUT
    / "V0181_REPORT_SHA256.txt"
).write_text(
    f"{REPORT_SHA}  {REPORT_PATH.name}\n",
    encoding="utf-8",
)

print()
print("=" * 90)
print(
    "ARC-v0.18.1 CROSS-ENCODER SIGNED-DIRECTION AUDIT — COMPLETE"
)
print("Claim gate:", CLAIM_GATE)
print("Output:", OUT)
print("Report SHA-256:", REPORT_SHA)
print("Test accessed:", False)
print("=" * 90)
